# Diabetes Risk Prediction from Health-Survey Data

*Project overview and methodological framework (CRISP-DM)*

This notebook is the project's entry point: it states the objective, describes the data, and establishes CRISP-DM as the methodological framework that governs notebooks 01–08.

**Seminar:** Advanced Applied Data Science - Goethe University Frankfurt  
**Term:** Summer Semester 2026  
**Supervisor:** Prof. Dr. Kevin Bauer  
**Group:** Diabetes Prediction  

**Dataset:** CDC BRFSS 2015 — Diabetes Health Indicators  
[UCI ML Repository · Dataset #891](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)

---

## Team

- Hasher Malik — 7632048
- Jan Erdorf — 8748557
- Ilias El Ouali — 7632585
- Sophia Schaal — 7229428

---

## Project Overview

This project builds a binary classifier that predicts diabetes risk from self-reported health and lifestyle survey responses collected in the CDC BRFSS 2015 dataset. The goal is to support public-health screening programs by identifying high-risk individuals for follow-up clinical testing — without requiring laboratory results. Diabetes is also the leading driver of severe complications such as end-stage renal disease: in 2014, roughly 44 % of new ESRD cases in the U.S. and Puerto Rico were attributed to diabetes (Burrows et al., 2017), underscoring the public-health value of early risk identification. The project follows the **Cross-Industry Standard Process for Data Mining (CRISP-DM)**, progressing through business understanding, data exploration, preparation, modelling, and evaluation in a structured, iterative fashion. The formal success criteria, evaluation metrics, and error-cost structure are established in NB01.

## 1 · The Data

### Source

The **Behavioral Risk Factor Surveillance System (BRFSS)** is an annual, state-based, random-digit-dialed telephone health survey conducted by the U.S. Centers for Disease Control and Prevention (CDC). It is one of the largest continuously running health surveys in the world, tracking behavioral risk factors and chronic conditions across U.S. adults. The 2015 wave of this survey is the basis for this project. The **"CDC Diabetes Health Indicators"** subset (UCI ML Repository, dataset #891) is derived from BRFSS 2015 and can be loaded programmatically via the `ucimlrepo` package. The full BRFSS 2015 wave comprises 441,455 records across 330 variables; after cleaning and feature selection by A. Teboul (Kaggle), the curated subset used here retains 253,680 responses and 21 features.


**A note on provenance.** Two inconsistencies on the UCI landing page (#891) are worth clarifying, since the page is the link provided for this project. First, the **survey year is 2015**: the dataset's curator (A. Teboul) states the data is drawn from the BRFSS 2015 wave (441,455 records, 330 features), whereas the UCI page erroneously links the CDC *2014* survey documentation as its source — the curator's own description and the resulting `Diabetes_binary` variant confirm 2015. Second, the UCI prose describes *\"35 features … lab test results … three classes\"*, but this refers to the broader BRFSS source and Teboul's full set of derived variants. The resource actually served under `fetch_ucirepo(id=891)` — and used throughout this project — is the **tabular, binary variant with 21 categorical/integer features** and the binary target `Diabetes_binary`, as listed in the page's own variables table and metadata (Characteristics: *Tabular*; # Features: *21*). No laboratory measurements or unstructured fields are contained in this resource.

### The Dataset and Its Variants

The dataset and the binary target are prescribed by the seminar assignment: the CDC Diabetes Health Indicators set (UCI ML Repository #891), with the target *no diabetes* vs *prediabetes/diabetes*. Within the broader Teboul/Kaggle curation of BRFSS 2015, #891 corresponds to the **full, binary, imbalanced** variant — one of three published versions:

| Variant | Rows | Target | Balance |
|---|---|---|---|
| `diabetes_012_…` | 253,680 | 3 classes (0 none · 1 pre · 2 diabetes) | imbalanced |
| **`diabetes_binary_…`** (UCI #891, used here) | **253,680** | **binary (0/1)** | **~86 / 14** |
| `diabetes_binary_5050split_…` | 70,692 | binary (0/1) | balanced 50/50 |

Two properties of this prescribed variant shape the modelling approach:

1. **Binary target.** Prediabetes and diabetes are merged into one positive class; the distinction present in the 3-class file is not available here. This fits the screening framing — an *elevated-risk yes/no* flag for follow-up testing.
2. **Natural class imbalance (~14 % positive).** Unlike the down-sampled 50/50 file (70,692 rows), #891 preserves the real-world prevalence — exactly what is needed for a realistic PR-AUC baseline, threshold selection, and calibration. A balanced training sample would create a calibration mismatch against a 14 %-prevalence population, so we keep the imbalance and address it primarily through the choice of metric (PR-AUC) and a recall-driven decision threshold rather than resampling. Resampling and reweighting (SMOTE, `class_weight`) were evaluated in the modelling funnel but did not improve the final model and were not adopted; any in-fold resampling that was tested stayed strictly inside the cross-validation folds, leaving the held-out test set at true prevalence (see NB06 / NB07).

### Structure

| Property | Value |
|---|---|
| Samples | ~253,680 |
| Features | 21 |
| Target | `Diabetes_binary` (0 = No Diabetes, 1 = Prediabetes / Diabetes) |
| Class split | ~86 % negative / ~14 % positive |
| Missing values | None |

Feature types: 14 binary, 4 ordinal (GenHlth, Age, Education, Income), 2 count (MentHlth, PhysHlth), 1 continuous (BMI).

### Survey Nature and Coarse Coding

All variables are self-reported responses to a telephone survey. Several continuous quantities (age, income, education, general health) are binned into ordinal scales. BMI is reported by respondents and not clinically measured. These properties limit the precision of individual features but are representative of the information available in real-world public-health screening scenarios.

### Key Challenges
This dataset has three characteristics that shape the modelling approach, each handled at the relevant pipeline stage. **Class imbalance:** only ~14 % of records are positive, so accuracy is misleading; the imbalance is addressed through PR-AUC as the primary metric and a recall-driven threshold, with resampling evaluated but not adopted in the final model (NB06–07). **Label noise:** the target records whether a respondent was *told by a doctor* that they have diabetes — diagnosis status, not disease status — so undiagnosed individuals appear as negatives, producing asymmetric label noise on the negative class (established in NB01, revisited via false-positive profiling in NB08). This diagnosis-based definition mirrors how CDC surveillance itself treats the diabetic population: estimates rest on self-reported diagnosis, highly accurate for *diagnosed* cases but undercounting the undiagnosed (Burrows et al., 2017). **Exact duplicates:** about 14 % of rows are exact duplicates (NB02) — largely coincidental profile collisions inherent to coarse survey features, not data errors. Rather than discard them, we use group-aware splitting: all rows sharing a feature vector are kept entirely on one side of the partition (group-based train/test split in NB03, `StratifiedGroupKFold` cross-validation in NB06/07). This preserves the full dataset and its real-world profile multiplicity while preventing the same profile from leaking across the train/test boundary.

## 2 · Methodological Framework — CRISP-DM

The project follows the **Cross-Industry Standard Process for Data Mining (CRISP-DM)**, a technology- and domain-neutral standard process model for applied data-mining and machine-learning projects that has become the de-facto industry standard since its introduction (Wirth & Hipp, 2000; Chapman et al., 2000). The notebook sequence of this project is a direct instantiation of CRISP-DM's six phases — data preparation and modelling each span several notebooks, and deployment is treated conceptually within the seminar scope (mapping below). A defining characteristic of CRISP-DM is that it is explicitly **iterative**: insights gained in later phases regularly feed back into earlier ones — modelling difficulties may prompt a return to data preparation, and evaluation findings may reopen business-understanding questions. The cycle repeats until the solution meets the defined success criteria.

The six phases are:

**1. Business Understanding.** Translate the real-world problem into a formal data-mining objective. Define success criteria, evaluation metrics, and the cost structure of different prediction errors from a domain perspective.

**2. Data Understanding.** Explore the available data, assess its quality, verify completeness, and develop a preliminary picture of relevant patterns and potential modelling challenges.

**3. Data Preparation.** Clean and transform the raw data; handle encoding, outliers, and any quality issues identified in the previous phase; construct the feature set; and produce a split suitable for unbiased model evaluation.

**4. Modelling.** Select candidate algorithms, build training pipelines, tune hyperparameters, and compare model performance using the metrics and cross-validation strategy defined in Business Understanding.

**5. Evaluation.** Assess the best model against the business success criteria on held-out test data. Verify that the solution genuinely satisfies the original objective — including any fairness or calibration requirements — before drawing conclusions.

**6. Deployment.** Plan or implement how the model is integrated into an operational environment so that stakeholders can act on its predictions.

The phases map onto the notebooks as follows:

| Phase | Notebook(s) |
|---|---|
| Business Understanding | NB01 |
| Data Understanding | NB02 |
| Data Preparation | NB03 & NB05 |
| Modelling | NB4 & NB06–07 |
| Evaluation | NB08 |
| Deployment | conceptual (seminar scope) |

The diagram below illustrates the CRISP-DM cycle, including the feedback loops between phases.

![CRISP-DM process cycle](assets/crispdm.png)

*Figure: The six phases of CRISP-DM. Diagram by Kenneth Jensen, [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/), via Wikimedia Commons ([source](https://commons.wikimedia.org/wiki/File:CRISP-DM_Process_Diagram.png)).*

## 3 · Reproducibility & How to Run

### Environment

Install all dependencies from the repository root:

```bash
pip install -r requirements.txt
```

Python 3.14 or later is required. Key packages include `scikit-learn`, `lightgbm`, `imbalanced-learn`, `shap`, and `ucimlrepo`.

### Random Seed

Every computational notebook declares `SEED = 42` at the top and passes it to all stochastic operations (train/test split, cross-validation, model initialisation, randomised hyperparameter search). No notebook introduces additional seeds.

### Data Access

Raw data is **not** stored in the repository. It is fetched automatically at runtime via the `ucimlrepo` package:

```python
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=891)
```

An internet connection is required for the first run.

### Run Order

Run the numbered notebooks in order from the project root. Each notebook depends on the outputs of those preceding it, so the sequence must be respected.

### Repository Layout

```
diabetes-prediction-ml/
├── notebooks/          # one notebook per CRISP-DM step (00–08)
│   └── assets/         # figures embedded in notebooks (e.g. crispdm.png)
├── literatur/          # cited papers (PDF)
├── src/
│   ├── features.py     # row-wise, leakage-free feature engineering (NB05+)
│   ├── inference.py    # standalone scoring of the final model (NB08 hand-off)
│   └── utils.py        # shared CV splitter + results ledger
├── data/               # git-ignored; generated at runtime
│   ├── raw/
│   └── processed/
├── models/             # git-ignored; written during modelling
├── outputs/            # git-ignored; plots/CSVs per notebook
├── CLAUDE.md
├── requirements.txt
├── README.md
└── .gitignore
```

The directories `data/`, `models/`, and `outputs/` are listed in `.gitignore` and are not tracked by Git.

## References

Burrows, N. R., Hora, I., Geiss, L. S., Gregg, E. W., & Albright, A. (2017). Incidence of end-stage renal disease attributed to diabetes among persons with diagnosed diabetes — United States and Puerto Rico, 2000–2014. *MMWR. Morbidity and Mortality Weekly Report, 66*(43), 1165–1170. https://doi.org/10.15585/mmwr.mm6643a2

Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc.

U.S. Centers for Disease Control and Prevention (CDC). *Behavioral Risk Factor Surveillance System (BRFSS): 2015 Survey Data and Documentation*. Atlanta, GA: CDC. https://www.cdc.gov/brfss/annual_data/annual_2015.html

UCI Machine Learning Repository. *CDC Diabetes Health Indicators* (Dataset #891). Original curation by A. Teboul (Kaggle). https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators

Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. In *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining* (pp. 29–39). Manchester, UK.